# 11 - Policy targeting under a budget

Estimating an effect is not the same as deciding what to do. Treatment capacity
is finite: a clinic can enrol so many patients, a campaign can afford so many
contacts. The decision is not *whether* the programme works on average but *who*
should receive it, and whether it is worth running at all.

This notebook turns the CATE estimates of notebook 03 into an assignment rule,
and evaluates it the way a decision would be evaluated — in units of realised
benefit, against both a random policy and the best any policy could do.

## Causal question

With capacity to treat only a fraction of the population, which individuals
should be treated, and how much better is a model-based ranking than treating at
random?

## Data and design

- **Unit of analysis:** one individual eligible for treatment.
- **Treatment:** `treatment`, binary and scarce — the budget binds.
- **Outcome:** `outcome`, continuous, higher is better.
- **Covariates:** `age`, `risk_score`, `prior_usage`, all observable at
  assignment time.
- **Ground truth:** `true_ite`, used *only* to evaluate policies, never to build
  them.

Keeping that last distinction is what makes the evaluation meaningful. The
ranking model sees exactly what a deployed system would see.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from causal_inference_lab.data_generators import make_heterogeneous_treatment_data
from causal_inference_lab.meta_learners import TMetaLearner

COVARIATES = ["age", "risk_score", "prior_usage"]

dataset = make_heterogeneous_treatment_data(n=5_000, seed=22)
data = dataset.data
features = data[COVARIATES]
true_effect = data["true_ite"].to_numpy()

print(f"population:      {len(data):,}")
print(f"average effect:  {dataset.true_ate:.3f}")
print(f"effect sd:       {true_effect.std():.3f}")
print(f"effect range:    {true_effect.min():.3f} to {true_effect.max():.3f}")
print()
print("If effects were constant, targeting could add nothing.")
print(f"They are not: the top decile averages {np.sort(true_effect)[-500:].mean():.3f} "
      f"against {np.sort(true_effect)[:500].mean():.3f} for the bottom.")

population:      5,000
average effect:  1.366
effect sd:       0.604
effect range:    -0.041 to 2.973

If effects were constant, targeting could add nothing.
They are not: the top decile averages 2.468 against 0.621 for the bottom.


**Interpretation.** The spread in individual effects is what creates the
opportunity. A programme with an average effect of 1.37 has a top decile
averaging 2.47 and a bottom decile averaging 0.62 — a fourfold difference — and
at the extreme it contains individuals for whom the effect is slightly negative.
Targeting is only ever as valuable as that variation is large; on a population
with constant effects, every policy below would perform identically.

## Estimand

Two quantities, and they are different.

The **CATE** is the statistical estimand: the expected effect given covariates.
It is what the model predicts.

The **policy value** is the decision estimand: the total benefit realised by
treating a chosen set, net of what treating them costs. It is what the decision
turns on, and a policy can have high value while its CATE estimates are
individually poor, because ranking needs only the *order* to be roughly right.

## Identification assumptions

1. **Conditional ignorability and overlap**, as in notebook 03 — the CATE model
   inherits every assumption of the estimator that produced it.
2. **No interference.** Treating one individual does not change another's
   outcome. Under a budget this is easy to violate: if capacity is rationed,
   treating one person may displace another.
3. **Transportability.** The relationship between covariates and effects holds
   in the population where the policy will be deployed. Ranking models degrade
   quietly when it does not.
4. **Stable effects over the evaluation window.** The effect a person would have
   had at assignment is the effect they get.

Assumption 3 is the one that fails most often in deployment, and the one this
notebook cannot check.

## Estimation

Build the ranking from observable data with a T-learner, then define three
policies at a fixed budget:

- **random** — the baseline any targeting must beat;
- **model** — treat the highest predicted effects;
- **oracle** — treat the highest *true* effects, which is unattainable and
  serves as the ceiling.

In [2]:
model = TMetaLearner().fit(
    data,
    covariates=COVARIATES,
    treatment_col="treatment",
    outcome_col="outcome",
)
predicted_effect = model.predict_cate(features)

print(f"correlation between predicted and true effect: "
      f"{np.corrcoef(predicted_effect, true_effect)[0, 1]:.3f}")


def policies(budget: int, seed: int = 11) -> dict[str, np.ndarray]:
    """Index arrays for the units each policy would treat."""
    rng = np.random.default_rng(seed)
    return {
        "random": rng.choice(len(data), size=budget, replace=False),
        "model": np.argsort(-predicted_effect)[:budget],
        "oracle": np.argsort(-true_effect)[:budget],
    }


budget = int(0.2 * len(data))
selected = policies(budget)

print(f"\nbudget: treat {budget:,} of {len(data):,} ({budget / len(data):.0%})")
print()
for name, chosen in selected.items():
    print(f"  {name:7s} average true effect among treated: {true_effect[chosen].mean():.3f}")

correlation between predicted and true effect: 0.793

budget: treat 1,000 of 5,000 (20%)

  random  average true effect among treated: 1.344
  model   average true effect among treated: 2.225
  oracle  average true effect among treated: 2.339


**Interpretation.** Random targeting realises 1.344 per treated individual —
essentially the population average of 1.366, as it must. The model realises
2.225, and the oracle 2.339.

The model captures most of the distance between random and perfect despite a
predicted-versus-true correlation of only 0.79. That gap between modest
predictive accuracy and strong policy value is the central fact about targeting:
the decision needs the ranking to be approximately right, not the individual
predictions to be precise.

## Diagnostics

The natural diagnostic for a policy is not accuracy but the fraction of
*achievable* gain it captures — where achievable means the distance between
random and oracle. We check that this holds across budgets rather than at one
convenient point.

In [3]:
rows = []
for fraction in (0.05, 0.1, 0.2, 0.5, 0.8):
    size = int(fraction * len(data))
    chosen = policies(size)
    gains = {name: true_effect[idx].sum() for name, idx in chosen.items()}
    achievable = gains["oracle"] - gains["random"]
    rows.append(
        {
            "budget": f"{fraction:.0%}",
            "random": gains["random"] / size,
            "model": gains["model"] / size,
            "oracle": gains["oracle"] / size,
            "gain captured": (gains["model"] - gains["random"]) / achievable,
        }
    )

table = pd.DataFrame(rows)
print(table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

budget  random  model  oracle  gain captured
    5%   1.392  2.386   2.565          0.848
   10%   1.339  2.353   2.468          0.898
   20%   1.344  2.225   2.339          0.885
   50%   1.360  1.786   1.845          0.879
   80%   1.363  1.490   1.528          0.767


**Interpretation.** The model captures between 77% and 90% of the achievable
gain across budgets from 5% to 80%, so the result is not an artefact of one
capacity choice. The weakest point is the widest budget: at 80% capacity the
model captures 77%, because by then the policy is forced to include nearly
everyone and the remaining discrimination is among individuals whose effects
genuinely are similar.

The per-treated averages fall as the budget widens — 2.35 at a 10% budget, 1.79
at 50% — which is exactly right and worth stating explicitly: as capacity grows,
targeting reaches further down the effect distribution, and the marginal treated
individual is worth less. The value of targeting shrinks as the budget
approaches the whole population, where all three policies coincide.

## Uncertainty

The decision that actually matters is not which ranking to use but whether to
run the programme at all. That depends on the cost per treatment, which the
analysis so far has ignored entirely.

In [4]:
budget = int(0.2 * len(data))
selected = policies(budget)

rows = []
for cost in (0.5, 1.0, 1.4, 2.0, 2.5):
    # Cost is a label, not a quantity to be formatted alongside the money columns.
    row = {"cost per treatment": f"{cost:.2f}"}
    for name, chosen in selected.items():
        row[f"{name} (20% budget)"] = true_effect[chosen].sum() - cost * budget
    row["treat everyone"] = true_effect.sum() - cost * len(data)
    rows.append(row)

net = pd.DataFrame(rows)
print(net.to_string(index=False, float_format=lambda v: f"{v:,.0f}"))

print(f"\nbreak-even cost, treat everyone:      {true_effect.mean():.3f}")
print(f"break-even cost, model at 20% budget: {true_effect[selected['model']].mean():.3f}")

cost per treatment  random (20% budget)  model (20% budget)  oracle (20% budget)  treat everyone
              0.50                  844               1,725                1,839           4,329
              1.00                  344               1,225                1,339           1,829
              1.40                  -56                 825                  939            -171
              2.00                 -656                 225                  339          -3,171
              2.50               -1,156                -275                 -161          -5,671

break-even cost, treat everyone:      1.366
break-even cost, model at 20% budget: 2.225


**Interpretation.** The policy ranking reverses with cost, and this is the most
consequential result in the notebook.

At a cost of 0.5, treating everyone is best by a wide margin: the programme is
cheap enough that even low-benefit individuals are worth treating, and
restricting to 20% forfeits value. At a cost of 2.0, treating everyone destroys
value — a large negative — while targeted treatment remains clearly positive.

The break-even costs make the boundary explicit. Universal treatment pays only
below 1.366; targeted treatment pays up to 2.225. Between those two numbers lies
a range where the programme is worth running *only if it is targeted*. A
cost-effectiveness analysis that evaluated the programme universally would
recommend cancelling it, and would be wrong.

## Limitations

- **Evaluated on the data that trained the ranking.** The model scores and is
  scored on the same individuals, which flatters it. A deployed system faces new
  people, and honest evaluation needs a holdout.
- **Ground truth is unavailable in practice.** Every number here uses
  `true_ite`. Real policy evaluation must rely on a holdout experiment or an
  uplift-based estimator, both noisier than what is shown.
- **No uncertainty on the policy value.** The comparisons are point estimates.
  Two policies differing by less than sampling noise are not distinguishable,
  and nothing here reports that threshold.
- **Costs are assumed constant and known.** Per-treatment cost is rarely uniform
  across individuals, and it interacts with who is selected.
- **Interference is assumed away**, which is uncomfortable under a binding
  budget: rationing is itself a mechanism by which one person's treatment
  affects another's.
- **Fairness is not examined.** A ranking trained on historical data inherits
  whatever inequities shaped that data, and can concentrate treatment away from
  groups that were previously underserved. Effect-based targeting is not
  automatically equitable targeting, and this notebook does not audit it.